<div align="center">
<a href="https://rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/RapidFire - Blue bug -white text.svg" width="115"></a>
<a href="https://discord.gg/6vSTtncKNN"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/discord-button.svg" width="145"></a>
<a href="https://oss-docs.rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/documentation-button.svg" width="125"></a>
<br/>
Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/RapidFireAI/rapidfireai">GitHub</a></i> ⭐
<br/>
👉 <b>Note:</b> This Colab notebook illustrates simplified usage of <code>rapidfireai</code>. For the full RapidFire AI experience with advanced experiment manager, UI, and production features, see our <a href=\"https://oss-docs.rapidfire.ai/en/latest/walkthrough.html\">Install and Get Started</a> guide.
<br/>
🎬 Watch our <a href=\"https://youtu.be/vVXorey0ANk\">intro video</a> to get started!
</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RapidFireAI/rapidfireai/blob/main/tutorial_notebooks/rag-contexteng/rf-colab-rag-fiqa-tutorial.ipynb)

⚠️ **IMPORTANT:** Do not let the Colab notebook tab stay idle for more than 5 min; Colab will disconnect otherwise. Interact with the cells to avoid disconnection.

# Optimizing RAG Pipelines with RapidFire AI

Retrieval-Augmented Generation (RAG) is a practical way to make an AI assistant **answer using your documents**:

- **Retrieve**: find the most relevant passages for a question.
- **Generate**: give those passages to a language model so it can answer *grounded in evidence*.

In this beginner-friendly Colab, we’ll build and evaluate a RAG pipeline for a **financial opinion Q&A** assistant using the [FiQA dataset](https://huggingface.co/datasets/explodinggradients/fiqa).

Examples of the kind of questions we’re targeting:

- “Should I invest in index funds or individual stocks?”
- “What’s a good way to save for retirement in my 30s?”
- “Is it worth refinancing my mortgage right now?”

## What We’re Building

A concrete RAG pipeline that looks like this:

1. **Load a financial corpus** (documents + posts).
2. **Split documents into chunks** (so we can search smaller, more relevant pieces).
3. **Embed the chunks** (turn text into vectors) and store them in a vector index (FAISS).
4. **Retrieve top‑K chunks** for each question using similarity search.
5. *(Optional)* **Rerank** the retrieved chunks with a stronger model to keep only the best evidence.
6. **Build a prompt** that includes the question + retrieved context.
7. **Generate an answer** with a small vLLM model.
8. **Evaluate retrieval quality** (Precision, Recall, NDCG@5, MRR) so we can tell which settings find better evidence.

## Our Approach

RAG has a lot of “knobs”, and it’s easy to lose track of what helped. In this notebook we’ll focus on **retrieval quality** by keeping the generator (the vLLM model) fixed and only varying retrieval settings.

We’ll use [RapidFireAI](https://github.com/RapidFireAI/rapidfireai) to:

- **Define a small retrieval grid**: 2 chunking strategies × 2 reranking `top_n` values = **4 retrieval configs**.
- **Run all configs the same way** on the same dataset.
- **Compare retrieval metrics side-by-side** as they update (Precision/Recall/NDCG/MRR) to pick the best evidence-finding setup.

## Install RapidFire AI Package and Setup
### Option 1: Install Locally (or on a VM)
For the full RapidFire AI experience—advanced experiment management, UI, and production features—we recommend installing the package on a machine you control (for example, a VM or your local machine) rather than Google Colab. See our [Install and Get Started](https://oss-docs.rapidfire.ai/en/latest/walkthrough.html) guide.

### Option 2: Install in Google Colab
For simplicity, you can run this notebook on Google Colab. This notebook is configured to run end-to-end on Colab with no local installation required.

In [1]:
try:
    import rapidfireai
    print("✅ rapidfireai already installed")
except ImportError:
    %pip install rapidfireai==0.15.1  # Takes 1 min
    !rapidfireai init --evals # Takes 1 min

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 7.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of trackio to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of trackio[gpu] to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 113.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.8/773.8 kB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 70.5 MB/s eta 0:0

### Import RapidFire Components

Import RapidFire’s core classes for defining the RAG pipeline and running a small retrieval grid search (plus a Colab-friendly protobuf setting).

In [2]:
import os
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'

from rapidfireai import Experiment
from rapidfireai.automl import List, RFLangChainRagSpec, RFvLLMModelConfig, RFPromptManager, RFGridSearch
import re, json
from typing import List as listtype, Dict, Any

# If you get "AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'" from Colab, just rerun this cell

### Loading the Data

We load the FiQA **queries** and **relevance labels (qrels)**, then downsample to keep this Colab run fast.
Next, we filter the corpus to only documents relevant to the sampled queries and write a smaller `corpus_sampled.jsonl`.
Finally, we update `qrels` to match the sampled subset so evaluation stays consistent.

In [3]:
from datasets import load_dataset
import pandas as pd
import json, random
from pathlib import Path

# Dataset directory
dataset_dir = Path("/content/tutorial_notebooks/rag-contexteng/datasets")

# Load full dataset
fiqa_dataset = load_dataset("json", data_files=str(dataset_dir / "fiqa" / "queries.jsonl"), split="train")
fiqa_dataset = fiqa_dataset.rename_columns({"text": "query", "_id": "query_id"})
qrels = pd.read_csv(str(dataset_dir / "fiqa" / "qrels.tsv"), sep="\t")
qrels = qrels.rename(
    columns={"query-id": "query_id", "corpus-id": "corpus_id", "score": "relevance"}
)

# Downsample queries and corpus JOINTLY
sample_fraction = 0.001  # low sample_fraction makes demo faster but degrades metrics; set to 1.0 for full evals if running on a local machine.
rseed = 1
random.seed(rseed)

# Step 1: Sample queries
sample_size = int(len(fiqa_dataset) * sample_fraction)
fiqa_dataset = fiqa_dataset.shuffle(seed=rseed).select(range(sample_size))

# Convert query_ids to integers for matching
query_ids = set([int(qid) for qid in fiqa_dataset["query_id"]])

# Step 2: Get all corpus docs relevant to sampled queries
qrels_filtered = qrels[qrels["query_id"].isin(query_ids)]
relevant_corpus_ids = set(qrels_filtered["corpus_id"].tolist())

print(f"Using {len(fiqa_dataset)} queries")
print(f"Found {len(relevant_corpus_ids)} relevant documents for these queries")

# Step 3: Load corpus and filter to relevant docs
input_file = dataset_dir / "fiqa" / "corpus.jsonl"
output_file = dataset_dir / "fiqa" / "corpus_sampled.jsonl"

with open(input_file, 'r') as f:
    all_corpus = [json.loads(line) for line in f]

# Keep only relevant documents (convert _id to int for matching)
sampled_corpus = [doc for doc in all_corpus if int(doc["_id"]) in relevant_corpus_ids]

# Write sampled corpus
with open(output_file, 'w') as f:
    for doc in sampled_corpus:
        f.write(json.dumps(doc) + '\n')

print(f"Sampled {len(sampled_corpus)} documents from {len(all_corpus)} total")
print(f"Saved to: {output_file}")
print(f"Filtered qrels to {len(qrels_filtered)} relevance judgments")

# Update qrels to match
qrels = qrels_filtered

Generating train split: 0 examples [00:00, ? examples/s]

Using 6 queries
Found 16 relevant documents for these queries
Sampled 16 documents from 57638 total
Saved to: /content/tutorial_notebooks/rag-contexteng/datasets/fiqa/corpus_sampled.jsonl
Filtered qrels to 16 relevance judgments


### Defining the RAG Search Space
This is where RapidFireAI shines. Instead of hardcoding a single RAG configuration, we define a search space using RFLangChainRagSpec.

We will test:

* **2 Chunking Strategies**: Different chunk sizes (256 vs 128).
* **2 Reranking Strategies**: Different `top_n` values (2 vs 5).

This gives us 4 combinations to evaluate for the retrieval part.

In [4]:
from langchain_community.document_loaders import DirectoryLoader, JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# Per-Actor batch size for hardware efficiency
batch_size = 50

# 2 chunk sizes x 2 reranking top-n = 4 combinations in total
rag_gpu = RFLangChainRagSpec(
    document_loader=DirectoryLoader(
        path=str(dataset_dir / "fiqa"),
        glob="corpus_sampled.jsonl",
        loader_cls=JSONLoader,
        loader_kwargs={
            "jq_schema": ".",
            "content_key": "text",
            "metadata_func": lambda record, metadata: {
                "corpus_id": int(record.get("_id"))
            },  # store the document id
            "json_lines": True,
            "text_content": False,
        },
        sample_seed=42,
    ),
    # 2 chunking strategies with different chunk sizes
    text_splitter=List([
            RecursiveCharacterTextSplitter.from_tiktoken_encoder(
                encoding_name="gpt2", chunk_size=256, chunk_overlap=32
            ),
            RecursiveCharacterTextSplitter.from_tiktoken_encoder(
                encoding_name="gpt2", chunk_size=128, chunk_overlap=32
            ),
        ],
    ),
    embedding_cfg={
        "class": HuggingFaceEmbeddings,
        "model_name": "sentence-transformers/all-MiniLM-L6-v2",
        "model_kwargs": {"device": "cuda:0"},
        "encode_kwargs": {"normalize_embeddings": True, "batch_size": batch_size},
    },
    # FAISS is an in-memory store and only works in create mode.
    vector_store_cfg={
        "type": "faiss",
    }, # if not set, uses FAISS by default
    search_cfg={
        "type": "similarity",
        "k": 8
    },
    # 2 reranking strategies with different top-n values
    reranker_cfg={
        "class": CrossEncoderReranker,
        "model_name": "cross-encoder/ms-marco-MiniLM-L6-v2",
        "model_kwargs": {"device": "cpu"},
        "top_n": List([2, 5]),
    },
    enable_gpu_search=True,
)

### Define Data Processing and Postprocessing Functions

We retrieve context for each question and turn it into LLM-ready prompts.
Then we attach the “ground truth” relevant documents from FiQA (`qrels`) so we can score retrieval quality later.

In [5]:
def sample_preprocess_fn(
    batch: Dict[str, listtype], rag: RFLangChainRagSpec, prompt_manager: RFPromptManager
) -> Dict[str, listtype]:
    """Function to prepare the final inputs given to the generator model"""

    INSTRUCTIONS = "Utilize your financial knowledge, give your answer or opinion to the input question or subject matter."

    # Perform batched retrieval over all queries; returns a list of lists of k documents per query
    all_context = rag.get_context(batch_queries=batch["query"], serialize=False)

    # Extract the retrieved document ids from the context
    retrieved_documents = [
        [doc.metadata["corpus_id"] for doc in docs] for docs in all_context
    ]

    # Serialize the retrieved documents into a single string per query using the default template
    serialized_context = rag.serialize_documents(all_context)
    batch["query_id"] = [int(query_id) for query_id in batch["query_id"]]

    # Each batch to contain conversational prompt, retrieved documents, and original 'query_id', 'query', 'metadata'
    return {
        "prompts": [
            [
                {"role": "system", "content": INSTRUCTIONS},
                {
                    "role": "user",
                    "content": f"Here is some relevant context:\n{context}. \nNow answer the following question using the context provided earlier:\n{question}",
                },
            ]
            for question, context in zip(batch["query"], serialized_context)
        ],
        "retrieved_documents": retrieved_documents,
        **batch,
    }


def sample_postprocess_fn(batch: Dict[str, listtype]) -> Dict[str, listtype]:
    """Function to postprocess outputs produced by generator model"""
    # Get ground truth documents for each query; can be done in preprocess_fn too but done here for clarity
    batch["ground_truth_documents"] = [
        qrels[qrels["query_id"] == query_id]["corpus_id"].tolist()
        for query_id in batch["query_id"]
    ]
    return batch

### Define Custom Eval Metrics Functions

The following helper methods compute standard retrieval metrics (Precision, Recall, F1, NDCG@5, MRR) from the retrieved vs. ground-truth document IDs.
We compute metrics per batch and then combine them across batches so each config gets one consistent score.

In [6]:
import math


def compute_ndcg_at_k(retrieved_docs: set, expected_docs: set, k=5):
    """Utility function to compute NDCG@k"""
    relevance = [1 if doc in expected_docs else 0 for doc in list(retrieved_docs)[:k]]
    dcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(relevance))

    # IDCG: perfect ranking limited by min(k, len(expected_docs))
    ideal_length = min(k, len(expected_docs))
    ideal_relevance = [3] * ideal_length + [0] * (k - ideal_length)
    idcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(ideal_relevance))

    return dcg / idcg if idcg > 0 else 0.0


def compute_rr(retrieved_docs: set, expected_docs: set):
    """Utility function to compute Reciprocal Rank (RR) for a single query"""
    rr = 0
    for i, retrieved_doc in enumerate(retrieved_docs):
        if retrieved_doc in expected_docs:
            rr = 1 / (i + 1)
            break
    return rr


def sample_compute_metrics_fn(batch: Dict[str, listtype]) -> Dict[str, Dict[str, Any]]:
    """Function to compute all eval metrics based on retrievals and/or generations"""

    true_positives, precisions, recalls, f1_scores, ndcgs, rrs = 0, [], [], [], [], []
    total_queries = len(batch["query"])

    for pred, gt in zip(batch["retrieved_documents"], batch["ground_truth_documents"]):
        expected_set = set(gt)
        retrieved_set = set(pred)

        true_positives = len(expected_set.intersection(retrieved_set))
        precision = true_positives / len(retrieved_set) if len(retrieved_set) > 0 else 0
        recall = true_positives / len(expected_set) if len(expected_set) > 0 else 0
        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall) > 0
            else 0
        )

        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)
        ndcgs.append(compute_ndcg_at_k(retrieved_set, expected_set, k=5))
        rrs.append(compute_rr(retrieved_set, expected_set))

    return {
        "Total": {"value": total_queries},
        "Precision": {"value": sum(precisions) / total_queries},
        "Recall": {"value": sum(recalls) / total_queries},
        "F1 Score": {"value": sum(f1_scores) / total_queries},
        "NDCG@5": {"value": sum(ndcgs) / total_queries},
        "MRR": {"value": sum(rrs) / total_queries},
    }


def sample_accumulate_metrics_fn(
    aggregated_metrics: Dict[str, listtype],
) -> Dict[str, Dict[str, Any]]:
    """Function to accumulate eval metrics across all batches"""

    num_queries_per_batch = [m["value"] for m in aggregated_metrics["Total"]]
    total_queries = sum(num_queries_per_batch)
    algebraic_metrics = ["Precision", "Recall", "F1 Score", "NDCG@5", "MRR"]

    return {
        "Total": {"value": total_queries},
        **{
            metric: {
                "value": sum(
                    m["value"] * queries
                    for m, queries in zip(
                        aggregated_metrics[metric], num_queries_per_batch
                    )
                )
                / total_queries,
                "is_algebraic": True,
                "value_range": (0, 1),
            }
            for metric in algebraic_metrics
        },
    }

### Define Partial Multi-Config Knobs for vLLM Generator part of RAG Pipeline using RapidFire AI Wrapper APIs

We pick a lightweight vLLM model and sampling settings that fit in Colab GPU memory.
Then we bundle the generator + our preprocessing/metrics functions into `config_set`, which RapidFire will run across the 4 retrieval configs.

In [7]:
vllm_config1 = RFvLLMModelConfig(
    model_config={
        "model": "Qwen/Qwen2.5-0.5B-Instruct",
        "dtype": "half",
        "gpu_memory_utilization": 0.25,
        "tensor_parallel_size": 1,
        "distributed_executor_backend": "mp",
        "enable_chunked_prefill": False,
        "enable_prefix_caching": False,
        "max_model_len": 3000,
        "disable_log_stats": True,  # Disable vLLM progress logging
        "enforce_eager": True,
        "disable_custom_all_reduce": True,
    },
    sampling_params={
        "temperature": 0.8,
        "top_p": 0.95,
        "max_tokens": 128,
    },
    rag=rag_gpu,
    prompt_manager=None,
)

batch_size = 3 # Smaller batch size for generation
config_set = {
    "vllm_config": vllm_config1,  # Only 1 generator, but it represents 4 full configs
    "batch_size": batch_size,
    "preprocess_fn": sample_preprocess_fn,
    "postprocess_fn": sample_postprocess_fn,
    "compute_metrics_fn": sample_compute_metrics_fn,
    "accumulate_metrics_fn": sample_accumulate_metrics_fn,
    "online_strategy_kwargs": {
        "strategy_name": "normal",
        "confidence_level": 0.95,
        "use_fpc": True,
    },
}

### Create Config Group

We create an `RFGridSearch` over `config_set`, producing **4 retrieval configs** (2 chunkers × 2 rerankers) to run and compare.



In [8]:
# Simple grid search across all config combinations: 4 total (2 chunkers × 2 rerankers)
config_group = RFGridSearch(config_set)

### Create Experiment

An `Experiment` is RapidFire’s top-level container for this notebook run: it groups configs/runs, saves artifacts, and tracks metrics under a unique name.
We set `mode="evals"` because we’re running evaluation (not training). See the docs: https://oss-docs.rapidfire.ai/en/latest/experiment.html#api-experiment


In [9]:
experiment = Experiment(experiment_name="exp1-fiqa-rag-colab", mode="evals")

Created directory for database at /content/rapidfireai/db
Experiment exp1-fiqa-rag-colab created with Experiment ID: 1 at /content/rapidfireai/rapidfire_experiments/exp1-fiqa-rag-colab
Created directory: /content/rapidfireai/logs/exp1-fiqa-rag-colab
🌐 Google Colab detected. Ray dashboard URL: https://8855-gpu-t4-s-kkb-usw1b0-3krzfa9brmkh6-b.us-west1-0.prod.colab.dev
🌐 Google Colab detected. Dispatcher URL: https://8851-gpu-t4-s-kkb-usw1b0-3krzfa9brmkh6-b.us-west1-0.prod.colab.dev


### Display Ray Dashboard (Optional)

Ray is the system RapidFire uses under the hood to run work in parallel; this cell simply embeds Ray’s dashboard below so we can monitor what’s running.

In [10]:
# Display the Ray dashboard in the Colab notebook
from google.colab import output
output.serve_kernel_port_as_iframe(8855)

<IPython.core.display.Javascript object>

### Run Multi-Config Evals + Launch Interactive Run Controller

Now we get to the main function for running multi-config evals. Two tables will appear below the run_evals cell:
- The first table will appear immediately. It lists all preprocessing/RAG sources.
- After a short while, the second table will appear. It lists all individual runs with their knobs and metrics that are updated in real-time via online aggregation showing both estimates and confidence intervals.

RapidFire AI also provides an Interactive Controller panel UI for Colab that lets you manage executing runs dynamically in real-time from the notebook:

- ⏹️ **Stop**: Gracefully stop a running config
- ▶️ **Resume**: Resume a stopped run
- 🗑️ **Delete**: Remove a run from this experiment
- 📋 **Clone**: Create a new run by editing the config dictionary of a parent run to try new knob values; optional warm start of parameters
- 🔄 **Refresh**: Update run status and metrics

In [11]:
# Launch evals of all RAG configs in the config_group with swap granularity of 4 chunks
# NB: If your machine has more than 1 GPU, set num_actors to that number
results = experiment.run_evals(
    config_group=config_group,
    dataset=fiqa_dataset,
    num_actors=1,
    num_shards=4,
    seed=42,
)

=== Preprocessing RAG Sources ===


RAG Source ID,Status,Duration,Details
1,Complete,50.8s,"FAISS, GPU"
2,Complete,50.8s,"FAISS, GPU"



=== Multi-Config Experiment Progress ===


Run ID,Model,Status,Progress,Conf. Interval,text_splitter,chunk_size,chunk_overlap,embedding_cfg.class,embedding_cfg.encode_kwargs.batch_size,embedding_cfg.encode_kwargs.normalize_embeddings,embedding_cfg.model_name,vector_store_cfg.type,search_cfg.k,search_cfg.type,reranker_cfg.class,reranker_cfg.model_name,reranker_cfg.top_n,sampling_params,model_config,F1 Score,MRR,NDCG@5,Precision,Processing Time,Recall,Samples Per Second,Samples Processed,Throughput,Total
1,Qwen/Qwen2.5-0.5B-Instruct,COMPLETED,4/4,0.000,RecursiveCharacterTextSplitter,256,32,HuggingFaceEmbeddings,50,True,sentence-transformers/all-MiniLM-L6-v2,faiss,8,similarity,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,2,"temp=0.8, top_p=0.95, max_tokens=128","dtype=half, gpu_memory_utilization=0.25, tensor_parallel_size=1, distributed_executor_backend=mp, enable_chunked_prefill=False, enable_prefix_caching=False, max_model_len=3000, disable_log_stats=True, enforce_eager=True, disable_custom_all_reduce=True","63.97% [63.97%, 63.97%]","83.33% [83.33%, 83.33%]","22.69% [22.69%, 22.69%]","75.00% [75.00%, 75.00%]",377.53 seconds,"62.22% [62.22%, 62.22%]",0.02,6,0.0/s,6
2,Qwen/Qwen2.5-0.5B-Instruct,COMPLETED,4/4,0.000,RecursiveCharacterTextSplitter,256,32,HuggingFaceEmbeddings,50,True,sentence-transformers/all-MiniLM-L6-v2,faiss,8,similarity,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,5,"temp=0.8, top_p=0.95, max_tokens=128","dtype=half, gpu_memory_utilization=0.25, tensor_parallel_size=1, distributed_executor_backend=mp, enable_chunked_prefill=False, enable_prefix_caching=False, max_model_len=3000, disable_log_stats=True, enforce_eager=True, disable_custom_all_reduce=True","54.37% [54.37%, 54.37%]","63.89% [63.89%, 63.89%]","21.39% [21.39%, 21.39%]","49.17% [49.17%, 49.17%]",152.18 seconds,"76.67% [76.67%, 76.67%]",0.04,6,0.1/s,6
3,Qwen/Qwen2.5-0.5B-Instruct,COMPLETED,4/4,0.000,RecursiveCharacterTextSplitter,128,32,HuggingFaceEmbeddings,50,True,sentence-transformers/all-MiniLM-L6-v2,faiss,8,similarity,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,2,"temp=0.8, top_p=0.95, max_tokens=128","dtype=half, gpu_memory_utilization=0.25, tensor_parallel_size=1, distributed_executor_backend=mp, enable_chunked_prefill=False, enable_prefix_caching=False, max_model_len=3000, disable_log_stats=True, enforce_eager=True, disable_custom_all_reduce=True","63.97% [63.97%, 63.97%]","83.33% [83.33%, 83.33%]","20.54% [20.54%, 20.54%]","83.33% [83.33%, 83.33%]",128.13 seconds,"53.89% [53.89%, 53.89%]",0.05,6,0.1/s,6
4,Qwen/Qwen2.5-0.5B-Instruct,COMPLETED,4/4,0.000,RecursiveCharacterTextSplitter,128,32,HuggingFaceEmbeddings,50,True,sentence-transformers/all-MiniLM-L6-v2,faiss,8,similarity,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,5,"temp=0.8, top_p=0.95, max_tokens=128","dtype=half, gpu_memory_utilization=0.25, tensor_parallel_size=1, distributed_executor_backend=mp, enable_chunked_prefill=False, enable_prefix_caching=False, max_model_len=3000, disable_log_stats=True, enforce_eager=True, disable_custom_all_reduce=True","65.21% [65.21%, 65.21%]","66.67% [66.67%, 66.67%]","23.30% [23.30%, 23.30%]","59.72% [59.72%, 59.72%]",119.21 seconds,"80.00% [80.00%, 80.00%]",0.05,6,0.1/s,6


### View Results

In [12]:
# Convert results dict to DataFrame
results_df = pd.DataFrame([
    {k: v['value'] if isinstance(v, dict) and 'value' in v else v for k, v in {**metrics_dict, 'run_id': run_id}.items()}
    for run_id, (_, metrics_dict) in results.items()
])

results_df

,run_id,model_name,text_splitter,chunk_size,chunk_overlap,embedding_cfg,vector_store_cfg,search_cfg,reranker_cfg,sampling_params,model_config,Samples Processed,Processing Time,Samples Per Second,Total,Precision,Recall,F1 Score,NDCG@5,MRR
0,1,Qwen/Qwen2.5-0.5B-Instruct,RecursiveCharacterTextSplitter,256,32,"{'class': 'HuggingFaceEmbeddings', 'model_name...",{'type': 'faiss'},"{'type': 'similarity', 'k': 8}","{'class': 'CrossEncoderReranker', 'model_name'...","{'temperature': 0.8, 'top_p': 0.95, 'max_token...","{'dtype': 'half', 'gpu_memory_utilization': 0....",6,377.53 seconds,0.02,6,0.750000,0.622222,0.639683,0.226882,0.833333
1,2,Qwen/Qwen2.5-0.5B-Instruct,RecursiveCharacterTextSplitter,256,32,"{'class': 'HuggingFaceEmbeddings', 'model_name...",{'type': 'faiss'},"{'type': 'similarity', 'k': 8}","{'class': 'CrossEncoderReranker', 'model_name'...","{'temperature': 0.8, 'top_p': 0.95, 'max_token...","{'dtype': 'half', 'gpu_memory_utilization': 0....",6,152.18 seconds,0.04,6,0.491667,0.766667,0.543651,0.213949,0.638889
2,3,Qwen/Qwen2.5-0.5B-Instruct,RecursiveCharacterTextSplitter,128,32,"{'class': 'HuggingFaceEmbeddings', 'model_name...",{'type': 'faiss'},"{'type': 'similarity', 'k': 8}","{'class': 'CrossEncoderReranker', 'model_name'...","{'temperature': 0.8, 'top_p': 0.95, 'max_token...","{'dtype': 'half', 'gpu_memory_utilization': 0....",6,128.13 seconds,0.05,6,0.833333,0.538889,0.639683,0.205390,0.833333
3,4,Qwen/Qwen2.5-0.5B-Instruct,RecursiveCharacterTextSplitter,128,32,"{'class': 'HuggingFaceEmbeddings', 'model_name...",{'type': 'faiss'},"{'type': 'similarity', 'k': 8}","{'class': 'CrossEncoderReranker', 'model_name'...","{'temperature': 0.8, 'top_p': 0.95, 'max_token...","{'dtype': 'half', 'gpu_memory_utilization': 0....",6,119.21 seconds,0.05,6,0.597222,0.800000,0.652116,0.232953,0.666667


### End Experiment

In [ ]:
from google.colab import output
from IPython.display import display, HTML

display(HTML('''
<button id="continue-btn" style="padding: 10px 20px; font-size: 16px;">Click to End Experiment</button>
'''))

# eval_js blocks until the Promise resolves
output.eval_js('''
new Promise((resolve) => {
    document.getElementById("continue-btn").onclick = () => {
        document.getElementById("continue-btn").disabled = true;
        document.getElementById("continue-btn").innerText = "Continuing...";
        resolve("clicked");
    };
})
''')

# Actually end the experiment after the button is clicked
experiment.end()
print("Done!")

### View RapidFire AI Log Files

In [ ]:
# Get the experiment-specific log file
log_file = experiment.get_log_file_path()

print(f"📄 Log File: {log_file}")
print()

if log_file.exists():
    print("=" * 80)
    print(f"Last 30 lines of {log_file.name}:")
    print("=" * 80)
    with open(log_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines[-30:]:
            print(line.rstrip())
else:
    print(f"❌ Log file not found: {log_file}")

### Conclusion

We built a simple Financial Q&A RAG pipeline and compared **4 retrieval configurations** (chunking × reranking) using standard retrieval metrics.

Optional ideas to explore later:
- Increase `sample_fraction` (or run locally) for more reliable results.
- Try additional retrieval knobs (e.g., embedding model, `k`, chunk overlap) and re-run the same evaluation loop.
